# Etapa 1 — Agrupamento: AGNES + DTW + Coeficiente de Silhueta

**Objetivo:** Calcular a matriz de dissimilaridade via *Dynamic Time Warping* (L2-DTW com banda de Sakoe-Chiba $r=7$ dias) sobre as 154 séries diárias padronizadas em Z-score, aplicar agrupamento hierárquico aglomerativo (AGNES) comparando as ligações `average` e `ward`, determinar objetivamente o número de grupos *k* pelo Coeficiente de Silhueta, avaliar pontos negativos e interpretar os perfis obtidos.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

RAIZ_PROJETO = Path.cwd().parent if Path.cwd().name == "Codigo" else Path.cwd()
if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

from Codigo.src.agrupamento import (
    INTERVALO_K,
    METODOS_LIGACAO,
    RAIO_SAKOE_CHIBA_DIAS,
    SEMENTE_PADRAO,
    avaliar_agnes_silhueta,
    calcular_ou_carregar_matriz_dtw,
    gerar_figuras_agrupamento,
    gerar_tabela_clientes_agrupamento,
    normalizar_zscore_por_serie,
    resumir_perfis_clusters,
    salvar_saidas_agrupamento,
)
from Codigo.src.preparacao import carregar_serie_tratada

np.random.seed(SEMENTE_PADRAO)
print(f"Raiz do projeto: {RAIZ_PROJETO}")

## 1. Carga dos Dados Tratados e Matriz DTW (com Cache)

In [ ]:
df_diario = carregar_serie_tratada(
    RAIZ_PROJETO / "Dados/tratado/consumo_diario_clientes_completos_2011_2014.parquet", freq="D"
)
df_horario = carregar_serie_tratada(
    RAIZ_PROJETO / "Dados/tratado/consumo_horario_clientes_completos_2011_2014.parquet", freq="h"
)

df_z = normalizar_zscore_por_serie(df_diario)
matriz_dtw, est_dtw = calcular_ou_carregar_matriz_dtw(
    df_z,
    raio=RAIO_SAKOE_CHIBA_DIAS,
    caminho_cache=RAIZ_PROJETO / "Resultados/matriz_dtw_diario_154.npy",
)
print(json.dumps(est_dtw, indent=2, ensure_ascii=False))

## 2. AGNES (`average` vs. `ward`) e Avaliação de Silhueta ($k=2..10$)

In [ ]:
nomes_clientes = list(df_diario.columns)
resultados_agnes = avaliar_agnes_silhueta(matriz_dtw, nomes_clientes, METODOS_LIGACAO, INTERVALO_K)

linhas_k = []
for metodo in METODOS_LIGACAO:
    for k in INTERVALO_K:
        av = resultados_agnes[metodo]["avaliacoes_k"][k]
        linhas_k.append({
            "ligacao": metodo,
            "k": k,
            "silhueta_media": av["silhueta_media"],
            "n_negativos": av["n_negativos"],
            "tamanhos": str(av["tamanhos_clusters"]),
        })
display(pd.DataFrame(linhas_k))

## 3. Perfis dos Clusters, Figuras e Persistência

In [ ]:
df_perfis = pd.concat([
    resumir_perfis_clusters(df_diario, df_horario, matriz_dtw, resultados_agnes["average"]["rotulos_por_k"][2], resultados_agnes["average"]["silhuetas_por_k"][2], "average_k2"),
    resumir_perfis_clusters(df_diario, df_horario, matriz_dtw, resultados_agnes["ward"]["rotulos_por_k"][3], resultados_agnes["ward"]["silhuetas_por_k"][3], "ward_k3"),
    resumir_perfis_clusters(df_diario, df_horario, matriz_dtw, resultados_agnes["ward"]["rotulos_por_k"][4], resultados_agnes["ward"]["silhuetas_por_k"][4], "ward_k4"),
], ignore_index=True)
display(df_perfis)

df_clientes_rotulos = gerar_tabela_clientes_agrupamento(df_diario, resultados_agnes)
figuras = gerar_figuras_agrupamento(df_diario, df_horario, resultados_agnes, RAIZ_PROJETO / "Documentacao/figuras")
figuras_rel = [str(Path(f).relative_to(RAIZ_PROJETO)) for f in figuras]
salvar_saidas_agrupamento(est_dtw, resultados_agnes, df_perfis, df_clientes_rotulos, figuras_rel, RAIZ_PROJETO / "Resultados")

for fig_path in figuras:
    display(Image(filename=fig_path))